# RFM Analysis — Customer Retention & Churn Risk Scoring
### Online Retail II Dataset

**Dataset:** online_retail_II.csv — 1M+ real transactions from a UK online retailer (2009–2011)

**Objective:** Apply RFM (Recency, Frequency, Monetary) analysis to rank and segment customers by their transaction behavior, identify churn risk tiers, and generate actionable retention scores.

**Techniques Used:** RFM Scoring, Quantile-based Binning, Composite Scoring, Segment Heatmaps  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 6)
sns.set_style("whitegrid")

print("Libraries loaded.")


## 1. Load Real Dataset — online_retail_II.csv
Transaction-level data with over 1 million rows. Each row is one line item on an invoice.
We clean cancellations (negative quantities), remove missing Customer IDs, and compute invoice-level totals.

In [ ]:
# Load real Online Retail II dataset
txn_df = pd.read_csv('online_retail_II.csv', dtype={'Invoice': str})

# Parse dates
txn_df['InvoiceDate'] = pd.to_datetime(txn_df['InvoiceDate'])

# Drop rows missing Customer ID
txn_df.dropna(subset=['Customer ID'], inplace=True)
txn_df['Customer ID'] = txn_df['Customer ID'].astype(int).astype(str)

# Remove cancellations (Invoice starting with 'C') and negative quantities/prices
txn_df = txn_df[~txn_df['Invoice'].str.startswith('C')]
txn_df = txn_df[(txn_df['Quantity'] > 0) & (txn_df['Price'] > 0)]

# Compute line-item revenue
txn_df['Revenue'] = txn_df['Quantity'] * txn_df['Price']

# Reference date = day after last transaction
ref_date = txn_df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f"Total transactions : {len(txn_df):,}")
print(f"Unique customers   : {txn_df['Customer ID'].nunique():,}")
print(f"Date range         : {txn_df['InvoiceDate'].min().date()} → {txn_df['InvoiceDate'].max().date()}")
print(f"Reference date     : {ref_date.date()}")
txn_df.head()


## 2. Computing RFM Metrics

| Metric | Definition | Business Signal |
|---|---|---|
| **Recency (R)** | Days since last transaction | Lower = more engaged |
| **Frequency (F)** | Total number of transactions | Higher = more loyal |
| **Monetary (M)** | Total spend over analysis period | Higher = more valuable |


In [ ]:
rfm = txn_df.groupby('Customer ID').agg(
    recency   = ('InvoiceDate', lambda x: (ref_date - x.max()).days),
    frequency = ('Invoice', 'nunique'),      # unique invoices = purchase occasions
    monetary  = ('Revenue', 'sum')
).reset_index()

rfm.rename(columns={'Customer ID': 'customer_id'}, inplace=True)
rfm['monetary'] = rfm['monetary'].round(2)

print("=== RFM Metrics Summary ===")
print(rfm[['recency','frequency','monetary']].describe().round(2))
rfm.head(10)


## 3. RFM Distributions — EDA


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RFM Metric Distributions', fontsize=15, fontweight='bold')

configs = [
    ('recency',   'Recency (Days Since Last Transaction)', 'cornflowerblue'),
    ('frequency', 'Frequency (Number of Invoices)',        'steelblue'),
    ('monetary',  'Monetary (Total Revenue £)',            'navy'),
]

for ax, (col, title, color) in zip(axes, configs):
    sns.histplot(rfm[col], ax=ax, bins=30, color=color, edgecolor='white')
    ax.axvline(rfm[col].median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Median: {rfm[col].median():,.0f}')
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. RFM Scoring (1–5 Quantile-Based)

Each metric is scored 1–5 using quintile binning:
- **Recency:** Lower days = higher score (more recent = better)
- **Frequency & Monetary:** Higher value = higher score


In [ ]:
def robust_score(series, ascending=True, bins=5):
    """Score 1-5 using percentile rank — handles duplicate values gracefully."""
    pct = series.rank(pct=True)
    score = (pct * bins).apply(lambda x: min(int(np.ceil(x)), bins)).clip(lower=1)
    if not ascending:
        score = bins + 1 - score
    return score.astype(int)

# Recency: lower days = more recent = higher score
rfm['R_score'] = robust_score(rfm['recency'], ascending=False)
# Frequency & Monetary: higher = better
rfm['F_score'] = robust_score(rfm['frequency'], ascending=True)
rfm['M_score'] = robust_score(rfm['monetary'], ascending=True)

# Composite RFM score (weighted: R=30%, F=30%, M=40%)
rfm['RFM_score'] = (
    rfm['R_score'] * 0.30 +
    rfm['F_score'] * 0.30 +
    rfm['M_score'] * 0.40
).round(2)

# Percentile rank for easier interpretation
rfm['RFM_percentile'] = rfm['RFM_score'].rank(pct=True).round(3)

print("Score distribution:")
print(rfm[['R_score','F_score','M_score','RFM_score']].describe().round(2))
rfm.head(10)


## 5. Customer Segmentation by RFM Score
Mapping RFM scores to business-meaningful retention tiers.


In [ ]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    score   = row['RFM_score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Low-Frequency'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'At-Risk High Value'    # 🔴 Most important for retention
    elif r <= 2 and f <= 2:
        return 'Churned / Dormant'
    elif r <= 3 and m >= 4:
        return 'Potential Loyalists'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print("=== Segment Distribution ===")
seg_summary = rfm.groupby('segment').agg(
    count          = ('customer_id', 'count'),
    avg_recency    = ('recency', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean'),
    avg_rfm_score  = ('RFM_score', 'mean')
).round(2).sort_values('avg_rfm_score', ascending=False)
print(seg_summary)


In [ ]:
# RFM Heatmap: Recency vs Frequency, colored by Monetary
pivot = rfm.pivot_table(values='monetary', index='R_score', columns='F_score', aggfunc='mean')

plt.figure(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'Avg Monetary Value (£)'})
plt.title('RFM Heatmap — Avg Monetary Value by Recency & Frequency Score',
          fontsize=13, fontweight='bold')
plt.xlabel('Frequency Score (1=Low, 5=High)')
plt.ylabel('Recency Score (1=Old, 5=Recent)')
plt.tight_layout()
plt.savefig('rfm_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Retention Priority Tier — Sales Action List
Generating a prioritized outreach list for the Field and Phone Sales teams.  
Focus: **At-Risk High Value** customers who are disengaging despite historically high spend.


In [ ]:
# Priority tier mapping
priority_map = {
    'At-Risk High Value'   : 'P1 — Immediate Save Action',
    'Champions'            : 'P2 — Proactive Retention',
    'Loyal Customers'      : 'P2 — Proactive Retention',
    'Potential Loyalists'  : 'P3 — Upsell / Engage',
    'Needs Attention'      : 'P3 — Re-engagement Campaign',
    'Recent Low-Frequency' : 'P4 — Onboarding / Activation',
    'Churned / Dormant'    : 'P5 — Win-back or Deprioritize',
}

rfm['priority_tier'] = rfm['segment'].map(priority_map)

# Sales action list — P1 customers
p1_list = rfm[rfm['priority_tier'] == 'P1 — Immediate Save Action'].sort_values(
    'monetary', ascending=False
)[['customer_id','recency','frequency','monetary','RFM_score','segment','priority_tier']]

print(f"P1 At-Risk High Value customers: {len(p1_list)}")
print("\nTop 15 by Monetary Value:")
print(p1_list.head(15).to_string(index=False))


In [ ]:
# Segment count bar chart
seg_counts = rfm['segment'].value_counts().sort_values()
colors_bar  = ['#d62728' if 'At-Risk' in s else '#1f77b4' for s in seg_counts.index]

plt.figure(figsize=(12, 6))
bars = plt.barh(seg_counts.index, seg_counts.values, color=colors_bar, edgecolor='white')
plt.xlabel('Number of Customers')
plt.title('Customer Count by RFM Segment', fontsize=14, fontweight='bold')

for bar, val in zip(bars, seg_counts.values):
    plt.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('segment_counts.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Revenue at risk from At-Risk customers
at_risk = rfm[rfm['segment'] == 'At-Risk High Value']
total_revenue   = rfm['monetary'].sum()
at_risk_revenue = at_risk['monetary'].sum()

print(f"Total Portfolio Revenue  : £{total_revenue:>15,.2f}")
print(f"At-Risk Segment Revenue  : £{at_risk_revenue:>15,.2f}")
print(f"Revenue at Risk          : {at_risk_revenue/total_revenue:.1%} of portfolio")
print(f"Customers at Risk        : {len(at_risk)} of {len(rfm)} ({len(at_risk)/len(rfm):.1%})")


## 7. Business Recommendations & Next Steps

| Priority | Segment | Action | Channel |
|---|---|---|---|
| **P1** | At-Risk High Value | Immediate outreach — offer retention incentives, payment flexibility | Phone Sales / Field RM |
| **P2** | Champions | Proactive loyalty program, credit limit increase, VIP benefits | Account Manager |
| **P2** | Loyal Customers | Rewards optimization, referral programs, cross-sell | Digital + Email |
| **P3** | Potential Loyalists | Spend activation offers, category-based incentives | Digital |
| **P4** | Recent Low-Frequency | Onboarding nudges, digital adoption campaigns | Email / App |
| **P5** | Churned / Dormant | Win-back campaign or portfolio cleanup | Automated |

**Key Metric to Track:** Charge Volume retained per P1 customer saved  
**Refresh Cadence:** Monthly (customer behavior evolves rapidly in commercial card portfolios)

**This RFM output can directly feed:**
- Field Sales prioritization dashboards (Power BI integration)
- Propensity model training data (churn labels from Churned/Dormant segment)
- A/B testing frameworks for treatment effectiveness measurement


In [ ]:
# Export full RFM output
rfm.to_csv('rfm_output.csv', index=False)
p1_list.to_csv('p1_at_risk_save_list.csv', index=False)

print(f"Exported rfm_output.csv          — {len(rfm)} customers")
print(f"Exported p1_at_risk_save_list.csv — {len(p1_list)} P1 customers")
